In [2]:
import requests
import pandas as pd

# Direct MapServer endpoint captured from your browser network request
endpoint = "https://tnmap.tn.gov/arcgis/rest/services/SAFETY/MapForDashboards/MapServer/0/query"

# SQL condition for both Fatal and Suspected Serious Injury crashes
where_clause = "Crash_Type IN ('Fatal', 'Suspected Serious Injury')"

# Requested fields matching your network request payload

all_records = []
offset = 0
record_limit = 1000

while True:
    params = {
        "where": where_clause,
        "outFields": "*",
        "f": "json",  # Overrides 'pbf' so python can parse standard JSON easily
        "resultOffset": offset,
        "resultRecordCount": record_limit,
        "orderByFields": "OBJECTID ASC",
        "returnGeometry": "false"  # Set to "true" if you need spatial coordinates
    }

    response = requests.get(endpoint, params=params)
    data = response.json()
    
    features = data.get("features", [])
    if not features:
        break

    # Extract attribute dictionaries
    records = [f["attributes"] for f in features]
    all_records.extend(records)
    
    # Break loop if we've fetched all available records
    if len(features) < record_limit:
        break
        
    offset += record_limit
    print(f"Fetched {len(all_records)} records so far...")

# Convert to DataFrame
df = pd.DataFrame(all_records)

# Convert Epoch timestamps (milliseconds) in Collision_Date to human-readable dates
if "Collision_Date" in df.columns:
    df["Collision_Date"] = pd.to_datetime(df["Collision_Date"], unit="ms")

# Save output to CSV
df.to_csv("tn_fatal_and_serious_injury_crashes.csv", index=False)
print(f"Successfully saved {len(df)} total crash records to tn_fatal_and_serious_injury_crashes.csv")

Fetched 1000 records so far...
Fetched 2000 records so far...
Fetched 3000 records so far...
Fetched 4000 records so far...
Fetched 5000 records so far...
Fetched 6000 records so far...
Fetched 7000 records so far...
Fetched 8000 records so far...
Fetched 9000 records so far...
Fetched 10000 records so far...
Fetched 11000 records so far...
Fetched 12000 records so far...
Fetched 13000 records so far...
Fetched 14000 records so far...
Fetched 15000 records so far...
Successfully saved 15126 total crash records to tn_fatal_and_serious_injury_crashes.csv


In [4]:
crash = pd.read_csv("tn_fatal_and_serious_injury_crashes.csv")

In [6]:
crash[crash['Crash_Type']=="Suspected Serious Injury"]

,OBJECTID,Agency_Name,City,Collision_Date,Collision_Hour,Collision_Month,Collision_Year,County,Crash_Type,DOW_Nmb,First_Harmfu_Event,Light_Condition,Weather,Work_Zone_Type
0,46,PULASKI POLICE DEPT,Pulaski,2024-01-03 14:55:00,14,1,2024,Giles,Suspected Serious Injury,4,Other Non-Fixed Object,Daylight,Clear,NaN
1,61,PULASKI POLICE DEPT,Pulaski,2024-01-04 12:50:00,12,1,2024,Giles,Suspected Serious Injury,5,Motor Vehicle-In-Transport On Same Roadway,Daylight,Cloudy,NaN
2,77,METROPOLITAN NASHVILLE POLICE DEPT,Nashville,2024-01-05 08:25:00,8,1,2024,Davidson,Suspected Serious Injury,6,Embankment Rock/Stone/Concrete,Daylight,Clear,NaN
3,87,METROPOLITAN NASHVILLE POLICE DEPT,Nashville,2024-01-05 17:39:00,17,1,2024,Davidson,Suspected Serious Injury,6,Pedestrian,Dusk,Clear,NaN
4,89,METROPOLITAN NASHVILLE POLICE DEPT,Nashville,2024-01-05 18:31:00,18,1,2024,Davidson,Suspected Serious Injury,6,Pedestrian,Dark-Lighted,Clear,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14990,119560,THP District 5 - Fall Branch,Mosheim,2026-08-26 06:28:00,6,8,2026,Greene,Suspected Serious Injury,4,Motor Vehicle-In-Transport On Same Roadway,Daylight,Clear,NaN
14991,119561,THP District 2 - Chattanooga,Not in City Limit,2026-08-25 19:34:00,19,8,2026,Grundy,Suspected Serious Injury,3,Pedestrian,Dawn,Clear,NaN
14992,119565,PORTLAND POLICE DEPT,Portland,2026-08-25 00:00:00,24,8,2026,Sumner,Suspected Serious Injury,3,Ditch,Dark-Not Lighted,Clear,NaN
14993,119583,THP District 5 - Fall Branch,Not in City Limit,2026-08-26 13:22:00,13,8,2026,Sullivan,Suspected Serious Injury,4,Other Animal,Daylight,Clear,NaN
